# 01 — Data Ingestion

Downloads earnings call transcripts from Motley Fool, parses them into speaker turns, and saves as structured JSON.

The parsing logic lives in `src/ingestion.py`. This notebook defines which transcripts to process and runs the pipeline.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.ingestion import process_transcript, save_transcript

## Transcripts to process

Each entry needs a Motley Fool URL, ticker, fiscal quarter, and date. These are the raw inputs to the pipeline — everything downstream (embeddings, retrieval, generation) depends on having clean, structured transcript data.

To find transcript URLs, search Google: `site:fool.com "COMPANY" "earnings call transcript" "Q? 202?"`

In [ ]:
TRANSCRIPTS = [
    {
        "url": "https://www.fool.com/earnings/call-transcripts/2025/08/01/apple-aapl-q3-2025-earnings-call-transcript/",
        "company": "AAPL",
        "quarter": "Q3-2025",
        "date": "2025-07-31",
    },
    {
        "url": "https://www.fool.com/earnings/call-transcripts/2025/10/31/apple-q4-2025-earnings-call-transcript/",
        "company": "AAPL",
        "quarter": "Q4-2025",
        "date": "2025-10-30",
    },
    {
        "url": "https://www.fool.com/earnings/call-transcripts/2026/01/29/apple-aapl-q1-2026-earnings-call-transcript/",
        "company": "AAPL",
        "quarter": "Q1-2026",
        "date": "2026-01-30",
    },
]

print(f"Transcripts to process: {len(TRANSCRIPTS)}")
for t in TRANSCRIPTS:
    print(f"  {t['company']} {t['quarter']} — {t['date']}")

## Download, parse, and save

`process_transcript()` handles the full pipeline for each URL: download the HTML, extract the article text, detect speaker turns (using the `Name:` pattern), and map speakers to roles from the Call Participants section. The output is a JSON file per transcript with structured speaker turns ready for embedding.

In [ ]:
all_transcripts = []

for t in TRANSCRIPTS:
    print(f"Processing {t['company']} {t['quarter']}...", end=" ")
    transcript = process_transcript(
        url=t["url"], company=t["company"],
        quarter=t["quarter"], date=t["date"],
    )
    path = save_transcript(transcript, output_dir="../data/processed")
    all_transcripts.append(transcript)
    print(f"{transcript['total_turns']} turns, {len(transcript['speakers'])} speakers")

total_turns = sum(t["total_turns"] for t in all_transcripts)
print(f"\nTotal: {len(all_transcripts)} transcripts, {total_turns} turns")

## Quick inspection

Sanity check that the parsing produced reasonable results — correct speakers, roles, and readable text. Catching issues here is much cheaper than debugging them in the retrieval or generation stages.

In [ ]:
# Preview first transcript
t = all_transcripts[0]
print(f"{t['company']} {t['quarter']} — {t['total_turns']} turns\n")
for turn in t["turns"][:3]:
    print(f"  {turn['speaker']} ({turn['role']}):")
    print(f"    {turn['text'][:120]}...\n")